In [1]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path(os.path.abspath(".."))
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = Path(os.path.abspath("."))

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / "data"
MODELS_DIR = PROJECT_ROOT / "models"
RESULTS_DIR = PROJECT_ROOT / "results"

In [2]:
RESEARCH_DATASET_DIR = (PROJECT_ROOT / "data" / "research_processed_smoke_auto").resolve()
RESEARCH_MODEL_DIR = (PROJECT_ROOT / "models" / "research_smoke_auto").resolve()
RESEARCH_RESULTS_DIR = (PROJECT_ROOT / "results" / "research_smoke_auto").resolve()

# 04 GPT

This notebook loads threshold-crossing anomaly candidates, builds compact GPT inputs, calls the OpenAI Responses API, parses structured JSON output, and compares AE-only vs AE+GPT decisions.

GPT is only used after anomaly score threshold crossing. The FiLM autoencoder remains the anomaly detector.

In [3]:
import matplotlib.pyplot as plt
import pandas as pd

from src.config import GPTConfig
from src.gpt_adjudicator import (
    adjudicate_anomaly,
    adjudicate_anomaly_records,
    build_window_summary,
    call_openai_responses_api,
    compare_ae_vs_gpt_decisions,
)
from src.utils import read_json

## Load Anomalous Windows Or Evaluation Results

In [4]:
cfg = GPTConfig(
    evaluation_dir=RESEARCH_RESULTS_DIR,
    output_dir=RESEARCH_RESULTS_DIR,
    max_records=100,
)

alerts = pd.read_csv(cfg.evaluation_dir / "realtime_alert_candidates.csv")
alerts = alerts.sort_values("anomaly_score", ascending=False).reset_index(drop=True)
alerts.head()

,window_id,container_id,machine_id,end_time,split,anomaly_score,threshold,score_over_threshold,feature_error_vector,top_feature_rank,...,predicted_label,event_id,realtime_step,alert_ready,gpt_triggered,gpt_label,gpt_severity,gpt_recommended_action,gpt_explanation,compact_anomaly_summary
0,38710,unknown_container,unknown_machine,38710,test,25.175177,4.342865,20.832312,"[0.4902978241443634, 0.11287228018045425, 0.50...","[3, 4, 2, 0, 7, 1, 6, 5]",...,1,-1,38710,True,True,critical,high,raise_alert,The score is far above threshold and the highe...,"{""window_id"": 38710, ""container_id"": ""unknown_..."
1,37870,unknown_container,unknown_machine,37870,test,24.337849,4.342865,19.994984,"[0.48756200075149536, 0.09929721057415009, 0.4...","[3, 4, 0, 2, 7, 1, 5, 6]",...,1,-1,37870,True,True,critical,high,raise_alert,The score is far above threshold and the highe...,"{""window_id"": 37870, ""container_id"": ""unknown_..."
2,37281,unknown_container,unknown_machine,37281,test,19.987984,4.342865,15.645119,"[0.5032023787498474, 0.12437083572149277, 0.51...","[3, 4, 2, 0, 7, 1, 5, 6]",...,1,-1,37281,True,True,critical,high,raise_alert,The score is far above threshold and the highe...,"{""window_id"": 37281, ""container_id"": ""unknown_..."
3,20421,unknown_container,unknown_machine,20421,test,19.766123,4.342865,15.423258,"[1.8485686779022217, 0.05683109909296036, 1.47...","[3, 4, 0, 2, 7, 1, 5, 6]",...,1,-1,20421,True,True,critical,high,raise_alert,The score is far above threshold and the highe...,"{""window_id"": 20421, ""container_id"": ""unknown_..."
4,18607,unknown_container,unknown_machine,18607,test,19.374149,4.342865,15.031284,"[1.9148701429367065, 0.04900514706969261, 0.80...","[3, 4, 0, 2, 7, 1, 5, 6]",...,1,-1,18607,True,True,critical,high,raise_alert,The score is far above threshold and the highe...,"{""window_id"": 18607, ""container_id"": ""unknown_..."


## Build Compact GPT Input

In [5]:
sample_payload = alerts.iloc[0].to_dict() if len(alerts) > 0 else {}
sample_summary = build_window_summary(sample_payload) if sample_payload else {}
sample_summary

{'window_id': 38710,
 'container_id': 'unknown_container',
 'machine_id': 'unknown_machine',
 'split': 'test',
 'time_range': {'start_time': -1, 'end_time': 38710},
 'anomaly_score': 25.1751766204834,
 'threshold': 4.342864990234375,
 'score_over_threshold': 20.832311630249023,
 'top_k_features': ['mem_gps', 'mpki', 'cpi', 'cpu_util', 'disk_io'],
 'top_k_feature_errors': [184.3610076904297,
  15.775625228881836,
  0.5050625205039978,
  0.4902978241443634,
  0.13868314027786255],
 'feature_error_vector': [0.4902978241443634,
  0.11287228018045425,
  0.5050625205039978,
  184.3610076904297,
  15.775625228881836,
  0.00871043000370264,
  0.009161935187876225,
  0.13868314027786255],
 'context_summary': {'container_app_du': 'unknown',
  'container_status': 'unknown',
  'machine_status': 'unknown',
  'machine_failure_domain_1': 'unknown',
  'machine_failure_domain_2': 'unknown'},
 'recent_logs': [],
 'recent_events': []}

## Call OpenAI Responses API

In [6]:
if sample_summary:
    sample_decision, sample_meta = call_openai_responses_api(sample_summary, cfg)
else:
    sample_decision, sample_meta = {}, {}

sample_meta, sample_decision

({'used_fallback': True,
  'reason': 'openai_request_failed',
  'error': "Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}"},
 {'label': 'critical',
  'severity': 'high',
  'explanation': 'The score is far above threshold and the highest reconstruction errors are in mem_gps, mpki, cpi. This is consistent with an active high-severity runtime issue.',
  'recommended_action': 'raise_alert'})

## Parse Structured JSON Output

In [7]:
sample_decision

{'label': 'critical',
 'severity': 'high',
 'explanation': 'The score is far above threshold and the highest reconstruction errors are in mem_gps, mpki, cpi. This is consistent with an active high-severity runtime issue.',
 'recommended_action': 'raise_alert'}

## Compare AE-only Vs AE+GPT Decisions

In [ ]:
adjudication_summary = adjudicate_anomaly_records(
    prediction_csv_path=cfg.evaluation_dir / "window_level_predictions.csv",
    config=cfg,
    max_records=cfg.max_records,
)
adjudication_summary

In [ ]:
adjudications = pd.read_csv(cfg.output_dir / "gpt_adjudications.csv")
comparison = pd.read_csv(cfg.output_dir / "ae_vs_gpt_comparison.csv")

adjudications.head(), comparison.head()

## Visualize GPT Decisions

In [ ]:
if len(adjudications) > 0:
    plt.figure(figsize=(6, 3))
    adjudications["severity"].value_counts().plot(kind="bar")
    plt.title("GPT Severity Distribution")
    plt.ylabel("Count")
    plt.show()

read_json(cfg.output_dir / "ae_vs_gpt_comparison.json")